<a href="https://colab.research.google.com/github/legna7816/ml-projects/blob/main/titanic/titanic_survival.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [53]:
# 1. 데이터 불러오기

# seaborn 내장 타이타닉 데이터
df = sns.load_dataset('titanic')

In [54]:
# 2. EDA - 모델 만들기 전 데이터 파악

df.info()          # 컬럼별 데이터 타입 + 결측치 개수 한눈에
df.isnull().sum()  # 컬럼별 결측치 개수 (deck/age/embarked에 많음)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [55]:
# 타겟(정답) 분포 확인 - 생존 0/1이 얼마나 균형잡혔는지
df['survived'].value_counts(normalize=True)

,proportion
survived,
0,0.616162
1,0.383838


In [56]:
# 가설 : 어떤 특징이 생존에 영향을 줄까?
df.groupby('sex')['survived'].mean()

,survived
sex,
female,0.742038
male,0.188908


In [57]:
df.groupby('pclass')['survived'].mean()

,survived
pclass,
1,0.629630
2,0.472826
3,0.242363


In [58]:
# 3. 전처리

# 3-1. 불필요/중복 컬럼 제거
df_clean = df.drop(columns=['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male']) # 다른 컬럼과 정보가 중복됨

In [59]:
# 3-2. 결측치 처리
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())    # 극단치(아기와 노인) 영향을 덜 받는 중앙값으로 채움
df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])    # 가장 흔한 값으로 채움

In [60]:
# 3-3. 범쥬형 -> 숫자 인코딩 (모델은 숫자만 이해)
df_clean['sex'] = df_clean['sex'].map({'male':0, 'female':1})

# embarked - C/Q/S 3개 카테고리 -> 원핫인코딩
# drop_first=True - 컬럼 2개(Q, S)만 생성, 둘 다 0이면 자동으로 C (중복 정보 방지)
df_clean = pd.get_dummies(df_clean, columns=['embarked'], drop_first=True)

In [61]:
# 3-4. Feature/Target 분리
X = df_clean.drop('survived', axis=1) # X: 입력 특징들
y = df_clean['survived']    # y : 맞춰야 할 정답

In [62]:
# 3-5. Feature Engineering (원본 컬럼 조합해 새 특징 생성)
X['family_size'] = X['sibsp'] + X['parch']           # 동승 가족 수
X['is_alone'] = (X['family_size'] == 0).astype(int)  # 혼자 탔으면 1, 아니면 0

# 'alone' 컬럼은 is_alone과 중복 정보 -> 제거
X = X.drop(columns=['alone'])

X.info() # 최종 확인 : 모든 컬럼 숫자형 + 결측치 0이어야 정상

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   pclass       891 non-null    int64  
 1   sex          891 non-null    int64  
 2   age          891 non-null    float64
 3   sibsp        891 non-null    int64  
 4   parch        891 non-null    int64  
 5   fare         891 non-null    float64
 6   embarked_Q   891 non-null    bool   
 7   embarked_S   891 non-null    bool   
 8   family_size  891 non-null    int64  
 9   is_alone     891 non-null    int64  
dtypes: bool(2), float64(2), int64(6)
memory usage: 57.6 KB


In [63]:
# 4. 모델 학습 / 평가
# 4-1. 학습용 / 평가용 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# test_size=0.2: 20%는 평가용으로 빼둠
# random_state=42: 매번 같은 방식으로 나뉘게 고정

In [64]:
# 4-2. 모델1: LogsticRegression
log_model = LogisticRegression(max_iter=200)

In [65]:
log_model.fit(X_train, y_train)       # 학습 (train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=200)

In [66]:
log_pred = log_model.predict(X_test)  # 예측 (test)

In [67]:
accuracy_score(y_test, log_pred)      # 정확도

0.7988826815642458

In [68]:
confusion_matrix(y_test, log_pred)    # 어떤 클래스를 헷갈렸는지

array([[89, 16],
       [20, 54]])

In [69]:
# 4-3. 모델2: RandomForest (DecisionTree 여러 개의 앙상블)
# 트리 하나는 과적합되기 쉬움 -> 여러 트리를 다르게 학습시켜 평균 -> 더 안정적

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
accuracy_score(y_test, rf_pred)

0.8324022346368715

In [70]:
# 5. 교차 검증 (Cross Validation)
# 데이터를 5번 다르게 나눠 5번 평가한 평균 비교
log_cv = cross_val_score(LogisticRegression(max_iter=200), X, y, cv=5)
tree_cv = cross_val_score(DecisionTreeClassifier(random_state=42), X, y, cv=5)
rf_cv = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5)

print('Logistic 평균:', log_cv.mean())
print('DecisionTree 평균', tree_cv.mean())
print('RandomForest 평균', rf_cv.mean())

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Logistic 평균: 0.792379637185362
DecisionTree 평균 0.7778231121712385
RandomForest 평균 0.8114619295712762


In [71]:
# -> 트리 개수를 늘린다고 무조건 좋아지지는 않음
rf_cv_test = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42), X, y, cv=5)
print('RandomForest (n_estimotors=300) 평균', rf_cv_test.mean())

RandomForest (n_estimotors=300) 평균 0.8058565061829137


In [72]:
# 6. 결과 해석 (Feature Importance)

# 모델이 어떤 특징을 보고 판단했는지 수치로 확인
importance = pd.Series(rf_model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

sex            0.271105
fare           0.262954
age            0.248771
pclass         0.077785
family_size    0.042982
sibsp          0.029954
embarked_S     0.024830
parch          0.020296
is_alone       0.012035
embarked_Q     0.009289
dtype: float64


In [73]:
# sex가 1위 -> EDA에서 세운 가설과 일치
# pclass가 없음 -> fare과 강하게 연관되어 정보가 겹침 = 다중공선성(multicollinearity)
# fare에 중요도가 물리며 pclass는 낮게 표시됨